# Manufactura con CatBoost: clasificación de piezas defectuosas

## Objetivo del ejercicio
En este ejercicio construiremos un modelo de **machine learning** para predecir si una pieza o lote será **defectuoso** (`1`) o **no defectuoso** (`0`) a partir de variables del proceso de manufactura.

## ¿Qué aprenderá el estudiante?
- Cómo cargar un dataset en Google Colab.
- Cómo explorar variables numéricas y categóricas.
- Cómo entrenar un modelo con **CatBoost**.
- Cómo evaluar un modelo de clasificación con métricas y visualizaciones.
- Cómo interpretar, de forma sencilla, los resultados del modelo.

## ¿Por qué usar CatBoost en este caso?
CatBoost es especialmente útil cuando el dataset mezcla **variables numéricas y categóricas**, porque:
- acepta variables categóricas de forma nativa;
- reduce el trabajo de codificación manual;
- suele ofrecer buen desempeño en datos tabulares;
- permite construir modelos competitivos con pocos ajustes iniciales.


## Ruta del ejercicio
1. Instalar e importar librerías.
2. Cargar el dataset desde la computadora del estudiante.
3. Realizar un análisis exploratorio breve.
4. Preparar los datos para el modelo.
5. Entrenar CatBoost.
6. Evaluar el desempeño con métricas y gráficas.
7. Interpretar las variables más importantes.
8. Cerrar con conclusiones sobre las ventajas de CatBoost.


## 1. Instalación e importación de librerías
En este bloque instalamos **CatBoost** y cargamos las librerías que usaremos para el análisis, la visualización, la partición de datos y la evaluación del modelo.


In [ ]:
%pip -q install catboost seaborn scikit-learn

import io
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from google.colab import files
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve,
)

sns.set_theme(style='whitegrid', palette='Blues')
pd.set_option('display.max_columns', None)

print('Librerías cargadas correctamente.')


## 2. Carga del dataset en Google Colab
Primero descargue el archivo `dataset_manufactura_catboost.csv` desde el portafolio.

Después, ejecute este bloque para **subir el CSV manualmente** desde su computadora. Este flujo ayuda a que el estudiante practique la carga de datos en Colab sin depender de rutas locales.


In [ ]:
print('Seleccione el archivo dataset_manufactura_catboost.csv desde su computadora.')
uploaded = files.upload()

if not uploaded:
    raise ValueError('No se cargó ningún archivo. Vuelva a ejecutar el bloque y seleccione el CSV.')

uploaded_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[uploaded_name]))

print(f'Archivo cargado: {uploaded_name}')
print(f'Registros: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')
df.head()


## 3. Vista general de los datos
Aquí revisamos el tamaño del dataset, los tipos de variables y si existen valores faltantes. Este paso permite entender la estructura de la información antes de construir el modelo.


In [ ]:
print('Dimensiones del dataset:', df.shape)
print('\nTipos de datos:')
print(df.dtypes)

print('\nValores faltantes por columna:')
print(df.isna().sum())


## 4. Distribución de la variable objetivo
La variable objetivo es `defectuoso`. Conviene revisar si las clases están balanceadas o si hay más observaciones de una categoría que de otra.


In [ ]:
target_counts = df['defectuoso'].value_counts().sort_index()
target_pct = df['defectuoso'].value_counts(normalize=True).sort_index() * 100

summary_target = pd.DataFrame({
    'conteo': target_counts,
    'porcentaje': target_pct.round(2)
})
summary_target.index = ['No defectuoso (0)', 'Defectuoso (1)']
summary_target


In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='defectuoso', hue='defectuoso', palette=['#9ecae1', '#3182bd'])
legend = ax.get_legend()
if legend is not None:
    legend.remove()
ax.set_title('Distribución de la variable objetivo')
ax.set_xlabel('Defectuoso')
ax.set_ylabel('Frecuencia')
ax.set_xticks([0, 1])
ax.set_xticklabels(['No', 'Sí'])
plt.show()


## 5. Exploración de variables numéricas
En manufactura, variables como temperatura, presión, velocidad, vibración y humedad pueden influir en la calidad. Primero las observamos con estadísticos descriptivos y después con histogramas.


In [ ]:
numeric_cols = ['temperatura', 'presion', 'velocidad_linea', 'vibracion', 'humedad']
df[numeric_cols].describe().T


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='#3182bd')
    axes[i].set_title(f'Distribución de {col}')

axes[-1].axis('off')
plt.tight_layout()
plt.show()


## 6. Comparación numérica según la clase
Los diagramas de caja ayudan a comparar cómo cambia cada variable numérica entre piezas defectuosas y no defectuosas.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, x='defectuoso', y=col, ax=axes[i], hue='defectuoso', palette=['#9ecae1', '#3182bd'])
    legend = axes[i].get_legend()
    if legend is not None:
        legend.remove()
    axes[i].set_title(f'{col} según defectuoso')
    axes[i].set_xlabel('Defectuoso')
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels(['No', 'Sí'])

axes[-1].axis('off')
plt.tight_layout()
plt.show()


## 7. Exploración de variables categóricas
Las variables `maquina`, `turno` y `material` son categóricas. CatBoost puede trabajarlas de forma nativa, lo cual es una de sus principales ventajas.


In [ ]:
categorical_cols = ['maquina', 'turno', 'material']

for col in categorical_cols:
    print(f'\nFrecuencias de {col}:')
    print(df[col].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, categorical_cols):
    sns.countplot(data=df, x=col, hue='defectuoso', ax=ax, palette=['#9ecae1', '#3182bd'])
    ax.set_title(f'{col} por clase')
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')
    ax.legend(title='Defectuoso', labels=['No', 'Sí'])

plt.tight_layout()
plt.show()


## 8. Correlación entre variables numéricas
Este mapa de calor solo se calcula con variables numéricas. Sirve para identificar relaciones lineales aproximadas entre variables del proceso.


In [ ]:
plt.figure(figsize=(8, 5))
correlation_matrix = df[numeric_cols + ['defectuoso']].corr(numeric_only=True)
sns.heatmap(correlation_matrix, annot=True, cmap='Blues', fmt='.2f')
plt.title('Mapa de correlación de variables numéricas')
plt.show()


## 9. Preparación de los datos
Ahora separamos las variables predictoras (`X`) y la variable objetivo (`y`). También identificamos las columnas categóricas para indicárselas a CatBoost.

Una ventaja importante es que **no necesitamos hacer one-hot encoding manual** en este ejemplo.


In [ ]:
X = df.drop(columns='defectuoso').copy()
y = df['defectuoso'].copy()

cat_features = [X.columns.get_loc(col) for col in categorical_cols]

print('Columnas predictoras:', X.columns.tolist())
print('Columnas categóricas:', categorical_cols)
print('Índices de columnas categóricas para CatBoost:', cat_features)


## 10. División en entrenamiento y prueba
Separamos los datos en entrenamiento y prueba para evaluar el modelo con observaciones que no participaron en el ajuste.

Usamos `stratify=y` para conservar la proporción de clases en ambos conjuntos.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print('Tamaño de entrenamiento:', X_train.shape)
print('Tamaño de prueba:', X_test.shape)
print('Distribución de la clase en entrenamiento:')
print(y_train.value_counts(normalize=True).round(3))
print('Distribución de la clase en prueba:')
print(y_test.value_counts(normalize=True).round(3))


## 11. Entrenamiento del modelo CatBoost
Entrenaremos un `CatBoostClassifier` con hiperparámetros simples y estables para principiantes.

- `iterations`: número de árboles.
- `depth`: profundidad máxima de cada árbol.
- `learning_rate`: velocidad de aprendizaje.
- `eval_metric`: métrica principal de seguimiento.
- `verbose=False`: evita una salida extensa durante el entrenamiento.


In [ ]:
train_pool = Pool(X_train, y_train, cat_features=cat_features)
test_pool = Pool(X_test, y_test, cat_features=cat_features)

model = CatBoostClassifier(
    iterations=250,
    depth=6,
    learning_rate=0.08,
    loss_function='Logloss',
    eval_metric='AUC',
    random_state=42,
    verbose=False,
)

model.fit(train_pool)
print('Modelo entrenado correctamente.')


## 12. Predicciones y probabilidades
Generamos dos tipos de salida:
- la clase predicha (`0` o `1`);
- la probabilidad estimada de que una pieza sea defectuosa.


In [ ]:
y_pred = model.predict(X_test).astype(int).ravel()
y_proba = model.predict_proba(X_test)[:, 1]

pred_preview = X_test.copy()
pred_preview['real'] = y_test.values
pred_preview['prediccion'] = y_pred
pred_preview['probabilidad_defectuoso'] = np.round(y_proba, 4)
pred_preview.head(10)


## 13. Métricas de evaluación
En clasificación no basta con una sola métrica. Aquí calculamos:
- **accuracy**: proporción de aciertos totales;
- **precision**: calidad de las predicciones positivas;
- **recall**: capacidad para detectar defectuosos reales;
- **F1-score**: equilibrio entre precisión y recall;
- **ROC AUC**: capacidad general para separar ambas clases.


In [ ]:
metrics_summary = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC AUC'],
    'Valor': [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0),
        roc_auc_score(y_test, y_proba),
    ]
})

metrics_summary['Valor'] = metrics_summary['Valor'].round(4)
metrics_summary


In [ ]:
print('Reporte de clasificación:')
print(classification_report(y_test, y_pred, target_names=['No defectuoso', 'Defectuoso'], zero_division=0))


## 14. Matriz de confusión
La matriz de confusión muestra en qué casos el modelo acierta y en cuáles se equivoca.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Pred. No', 'Pred. Sí'],
            yticklabels=['Real No', 'Real Sí'])
plt.title('Matriz de confusión')
plt.xlabel('Predicción')
plt.ylabel('Valor real')
plt.show()


## 15. Curva ROC
La curva ROC compara la tasa de verdaderos positivos contra la tasa de falsos positivos para distintos umbrales.


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_value = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'CatBoost (AUC = {auc_value:.3f})', color='#2171b5')
plt.plot([0, 1], [0, 1], '--', color='gray', label='Clasificador aleatorio')
plt.xlabel('Tasa de falsos positivos')
plt.ylabel('Tasa de verdaderos positivos')
plt.title('Curva ROC')
plt.legend()
plt.show()


## 16. Importancia de variables
Una forma sencilla de interpretar el modelo es revisar qué variables tuvieron mayor peso en las predicciones.


In [ ]:
feature_importance = pd.DataFrame({
    'variable': X.columns,
    'importancia': model.get_feature_importance()
}).sort_values('importancia', ascending=False)

feature_importance


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=feature_importance,
    x='importancia',
    y='variable',
    hue='variable',
    palette='Blues_r',
    legend=False
)
plt.title('Importancia de variables en CatBoost')
plt.xlabel('Importancia')
plt.ylabel('Variable')
plt.show()


## 17. Interpretación breve de resultados
Este bloque resume hallazgos iniciales del modelo y del dataset. El texto se genera a partir de las métricas y de la importancia de variables para facilitar la interpretación.


In [ ]:
top_features = feature_importance.head(3)['variable'].tolist()
positive_rate = y.mean() * 100

print('Interpretación general:')
print(f'- El dataset tiene {df.shape[0]} registros y {df.shape[1]} columnas.')
print(f'- La clase positiva (defectuoso = 1) representa aproximadamente el {positive_rate:.1f}% del total, por lo que existe un desbalance moderado.')
print(f'- El ROC AUC del modelo fue de {auc_value:.3f}, lo que ayuda a evaluar la capacidad de separación entre clases.')
print(f'- Las variables con mayor importancia fueron: {", ".join(top_features)}.')
print('- Esto no prueba causalidad, pero sí sugiere qué variables fueron más útiles para distinguir piezas defectuosas de piezas no defectuosas.')


## 18. Conclusiones

### ¿Qué ventaja mostró CatBoost en este ejercicio?
- Permitió trabajar con variables categóricas como `maquina`, `turno` y `material` sin codificación manual compleja.
- Ayudó a construir un flujo más simple para principiantes en problemas tabulares mixtos.
- Entregó métricas útiles con pocos ajustes iniciales.
- Facilitó la interpretación básica mediante la importancia de variables.

### Cierre
CatBoost es una excelente alternativa cuando se desea construir modelos sólidos para datos tabulares con mezcla de variables numéricas y categóricas. En contextos industriales y de manufactura puede ser una herramienta muy útil para apoyar decisiones de calidad, monitoreo y prevención de defectos.
